# IRAS 07546+3928 full-spectrum fitting with dynesty

This notebook fits the 4200–6800 Å rest-frame spectrum with the full additive GalSpec model used in `wrk_sagan_iras07546.ipynb`. It deliberately excludes the multiplicative polynomial. The Na D region (5770–6000 Å) is excluded from the likelihood.

The default configuration uses 200 live points, six worker processes, and runs until `dlogz=0.5`. On the development machine this required about 31 minutes with four cores and roughly 1.5 million likelihood evaluations.

In [ ]:
import os
from pathlib import Path

# Prevent every worker from also starting multiple BLAS threads.
os.environ.setdefault('OMP_NUM_THREADS', '1')

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.modeling import fitting, models
from extinction import ccm89, remove

import galspec

NLIVE = 200
N_PROCESSES = 10
DLOGZ = 0.05
RANDOM_SEED = 20260901

## Load and prepare the spectrum

In [ ]:
example_dir = Path.cwd()
if not (example_dir / 'data' / 'ir07546sed.fits').exists():
    example_dir = Path(galspec.__file__).resolve().parents[1] / 'example'
data_file = example_dir / 'data' / 'ir07546sed.fits'

with fits.open(data_file) as hdul:
    header = hdul[0].header
    flux = np.asarray(hdul[0].data, dtype=float) * 1e14

wave = header['CRVAL1'] + header['CD1_1'] * np.arange(flux.size)
flux = remove(ccm89(wave, 0.176, r_v=3.1, unit='aa'), flux)
wave /= 1.0 + 0.0953
use = (wave > 4200.0) & (wave < 6800.0)
wave, flux = wave[use], flux[use]
flux_error = np.maximum(0.05 * np.abs(flux), 1e-4)
fit_mask = ~((wave > 5770.0) & (wave < 6000.0))

print(f'{fit_mask.sum():,} fitted pixels out of {wave.size:,}')
fig, ax = plt.subplots(figsize=(12, 4))
ax.step(wave, flux, color='black', lw=0.5, where='mid')
ax.axvspan(5770, 6000, color='0.85', label='Excluded from likelihood')
ax.set(xlabel='Rest wavelength (Å)', ylabel='Scaled flux')
ax.legend();

## Construct the full additive model

In [ ]:
line_wave = galspec.utils.line_wave_dict
line_label = galspec.utils.line_label_dict

power_law = models.PowerLaw1D(amplitude=0.55, x_0=5500, alpha=1.0,
                                  fixed={'x_0': True}, name='Continuum')
iron = galspec.IronTemplate(amplitude=0.2, stddev=900, z=0, name='Fe II')
broad_ha = galspec.Line_MultiGauss(
    n_components=2, amp_c=2.22, dv_c=280, sigma_c=830,
    wavec=line_wave['Halpha'], name=line_label['Halpha'],
    amp_w0=0.26, dv_w0=-185, sigma_w0=2400)
broad_hb = galspec.Line_MultiGauss(
    n_components=2, amp_c=0.8, dv_c=-17, sigma_c=850,
    wavec=line_wave['Hbeta'], name=line_label['Hbeta'],
    amp_w0=0.3, dv_w0=-120, sigma_w0=2700,
    bounds={'sigma_w0': (100, 4000)})
broad_hg = galspec.Line_MultiGauss(
    n_components=1, amp_c=0.4, dv_c=80, sigma_c=1200,
    wavec=line_wave['Hgamma'], name=line_label['Hgamma'])
broad_he2 = galspec.Line_MultiGauss(
    n_components=1, amp_c=0.06, dv_c=85, sigma_c=2300,
    wavec=line_wave['HeII_4686'], name=line_label['HeII_4686'],
    bounds={'sigma_c': (100, 4000), 'dv_c': (-500, 500)})
oxygen = galspec.Line_MultiGauss_doublet(
    n_components=3, amp_c0=1.7, amp_c1=0.6, dv_c=-43, sigma_c=250,
    wavec0=line_wave['OIII_5007'], wavec1=line_wave['OIII_4959'], name='[O III]',
    amp_w0=0.35, dv_w0=-280, sigma_w0=340,
    amp_w1=0.05, dv_w1=500, sigma_w1=1000)
sulphur = galspec.Line_MultiGauss_doublet(
    n_components=1, amp_c0=0.1, amp_c1=0.1,
    wavec0=line_wave['SII_6716'], wavec1=line_wave['SII_6731'], name='[S II]')
narrow_ha = galspec.Line_Gaussian(amplitude=1.1, wavec=line_wave['Halpha'], name='narrow Ha')
narrow_hb = galspec.Line_Gaussian(amplitude=0.4, wavec=line_wave['Hbeta'], name='narrow Hb')
narrow_hg = galspec.Line_Gaussian(amplitude=0.12, wavec=line_wave['Hgamma'], name='narrow Hg')
narrow_he2 = galspec.Line_Gaussian(amplitude=0.08, wavec=line_wave['HeII_4686'], name='narrow He2')
narrow_o3 = galspec.Line_Gaussian(amplitude=0.05, wavec=line_wave['OIII_4363'], name='[O III] 4363')

model_initial = (power_law + iron + broad_ha + narrow_ha + broad_hb + narrow_hb
                 + broad_hg + narrow_hg + broad_he2 + narrow_he2
                 + oxygen + sulphur + narrow_o3)

oxygen.amp_c1.tied = galspec.tie_MultiGauss_doublet_ratio('[O III]', 2.98)
sulphur.sigma_c.tied = galspec.tie_MultiGauss_sigma_c('[O III]')
sulphur.dv_c.tied = galspec.tie_MultiGauss_dv_c('[O III]')
for line in (narrow_ha, narrow_hb, narrow_hg, narrow_he2, narrow_o3):
    line.sigma.tied = galspec.tie_MultiGauss_sigma_c('[O III]')
    line.dv.tied = galspec.tie_MultiGauss_dv_c('[O III]')

print('No multiplicative polynomial:', 'multi' not in model_initial.submodel_names)

## Deterministic initialization and finite priors

Nested sampling is initialized with the notebook's LevMar solution. The priors below are finite, physically constrained neighborhoods around that solution. Inspect and adjust them for a different scientific application.

In [ ]:
levmar = fitting.LevMarLSQFitter()
model_levmar = levmar(model_initial, wave, flux, weights=fit_mask.astype(float), maxiter=10000)

def make_local_bounds(model):
    result = {}
    for name in model.param_names:
        parameter = getattr(model, name)
        if parameter.fixed or parameter.tied:
            continue
        value = float(parameter.value)
        base = galspec.Dynesty_Fit._base_name(name).lower()
        if 'amplitude' in base or base.startswith('amp'):
            lower, upper = max(0, value * 0.35), max(value * 2, value + 0.05)
        elif base.startswith('sigma') or base == 'stddev':
            lower, upper = max(10, value * 0.5), max(30, value * 1.6)
        elif base.startswith('dv'):
            lower, upper = value - 500, value + 500
        elif base == 'alpha':
            lower, upper = value - 1.5, value + 1.5
        elif base in {'z', 'redshift'}:
            lower, upper = value - 0.003, value + 0.003
        else:
            width = max(abs(value) * 0.5, 0.1)
            lower, upper = value - width, value + width
        model_lower, model_upper = parameter.bounds
        if model_lower is not None and np.isfinite(model_lower):
            lower = max(lower, float(model_lower))
        if model_upper is not None and np.isfinite(model_upper):
            upper = min(upper, float(model_upper))
        result[name] = (lower, upper)
    return result

bounds = make_local_bounds(model_levmar)
print(f'{len(bounds)} free parameters')

## Run dynesty

`progress=True` displays dynesty's progress bar. No `maxiter` is supplied, so sampling continues until `dlogz=0.5`. On macOS, locally defined tie functions are supported through GalSpec's `multiprocess` pool.

In [ ]:
dynesty_fit = galspec.Dynesty_Fit(
    model_levmar, wave[fit_mask], flux[fit_mask], flux_error[fit_mask],
    bounds_dict=bounds, nlive=NLIVE, sample_method='rwalk', bound='multi',
    n_processes=N_PROCESSES, rstate=np.random.default_rng(RANDOM_SEED))

samples, _, parameter_names = dynesty_fit.fit(progress=True, dlogz=DLOGZ)
model_best, _, theta_best = dynesty_fit.get_best_fit()
print(f'Runtime: {dynesty_fit.runtime_seconds / 60:.2f} minutes')
print(f'Likelihood calls: {dynesty_fit.ncall:,}')
print(f'log Z = {dynesty_fit.log_evidence:.3f} ± {dynesty_fit.log_evidence_err:.3f}')

## Validate and display the fit

In [ ]:
model_flux = model_best(wave)
residual = (flux - model_flux) / flux_error
chi2 = np.sum(residual[fit_mask] ** 2)
reduced_chi2 = chi2 / (fit_mask.sum() - len(parameter_names))
print(f'Reduced chi-square: {reduced_chi2:.3f}')
print(f'Residual mean/std: {residual[fit_mask].mean():.3f} / {residual[fit_mask].std():.3f} sigma')

fig, (ax, axr) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
    gridspec_kw={'height_ratios': [3, 1]})
ax.step(wave, flux, color='0.3', lw=0.45, where='mid', label='Data')
ax.plot(wave, model_flux, color='tab:red', lw=1.0, label='Dynesty best fit')
ax.axvspan(5770, 6000, color='0.85', label='Excluded')
ax.set_ylabel('Scaled flux')
ax.legend()
axr.plot(wave, residual, color='0.3', lw=0.4)
axr.axhline(0, color='tab:red', lw=0.8)
axr.axvspan(5770, 6000, color='0.85')
axr.set(xlabel='Rest wavelength (Å)', ylabel='Residual / σ', ylim=(-8, 8))
fig.tight_layout();

In [ ]:
q16, q50, q84 = dynesty_fit.get_quantiles()
for name, median, low, high in zip(parameter_names, q50, q16, q84):
    print(f'{name:20s} = {median:12.5g} +{high-median:.3g} -{median-low:.3g}')